# SenGuard-MSC · Notebook 01 — Preprocessing and label protocol

Turns the deposited tables from notebook 00 into analysis-ready matrices, and **writes the analysis protocol before any MSC score exists**. External cohorts only get id harmonization and scale conversion here, with no score and no look at passage trends.

In [ ]:
import json, os, subprocess, sys, tempfile, textwrap
from pathlib import Path
import numpy as np, pandas as pd
ROOT_GUESS = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, EXT, DEV, ASC, BM
np.random.seed(sg.SEED)
for d in [RESULTS, sg.FIGS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)
sg.require(MANIFESTS / 'sample_manifest.csv', PROC / f'{DEV}_matrix_as_deposited.tsv.gz', PROC / f'{ASC}_matrix_as_deposited.tsv.gz', PROC / f'{BM}_matrix_as_deposited.tsv.gz',
           PROC / f'{BM}_gene_id_map.tsv', sg.EXT / 'wf-ml-markers-senescence', sg.EXT / 'HUSI')
print('ROOT', ROOT, '| pandas', pd.__version__, '| numpy', np.__version__)

## Development labels (GSE284666)
The authors' `assemble_anndata.py` defines the rule: treatments `P` and `EV` are controls and every other treatment is senescence-induced. The table below is computed from the GEO treatment text so any disagreement would show.

In [ ]:
manifest = sg.load_manifest()
labels = sg.dev_labels(manifest)
print(pd.crosstab(labels.treatment.str.replace(r'\s*\(.*', '', regex=True), labels.y).to_string())
print('\nSamples by cell type and label:')
print(pd.crosstab(labels.cell_line, labels.y).to_string())
print('\nTotal', len(labels), 'positives', int(labels.y.sum()), 'cell types', labels.cell_line.nunique())
out = labels[['title', 'cell_line', 'cell_type', 'treatment', 'label_candidate', 'y']].rename_axis('sample_id').reset_index()
out.to_csv(PROC / f'{DEV}_labels.csv', index=False)

## Normalization: reproduce the authors' method and verify it
`normalize_counts.py` filters genes and calls PyDESeq2's `fit_size_factors`. Here the same computation is implemented directly so it can be fit on training folds only. It is compared numerically with PyDESeq2 run in a separate environment. If that environment is absent the check is reported as not run.

In [ ]:
counts, ann = sg.load_as_deposited(DEV)
counts = counts.loc[labels.index]
filtered = sg.author_gene_filter(counts)
norm, sf, ref = sg.mor_normalize(filtered)
n_allpos = int((filtered.to_numpy() > 0).all(axis=0).sum())
print(f'{counts.shape[1]} genes -> {filtered.shape[1]} after author filter; genes non-zero in all samples: {n_allpos} (author adds +1 only if below 100)')
print('Size factors: min %.3f median %.3f max %.3f' % (sf.min(), sf.median(), sf.max()))

venv_py = ROOT / '.venv_check/Scripts/python.exe'
if not venv_py.exists():
    venv_py = ROOT / '.venv_check/bin/python'
check = {'status': 'not run: PyDESeq2 environment absent'}
if venv_py.exists():
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        filtered.to_csv(td / 'c.csv')
        code = textwrap.dedent('''
            import sys, json, pandas as pd, anndata as ad, pydeseq2
            from pydeseq2.dds import DeseqDataSet
            c = pd.read_csv(sys.argv[1], index_col=0).astype(int)
            a = ad.AnnData(X=c.values, obs=pd.DataFrame({'g': ['a'] * len(c)}, index=c.index.astype(str)), var=pd.DataFrame(index=c.columns.astype(str)))
            d = DeseqDataSet(adata=a, design='~1', quiet=True)
            d.fit_size_factors()
            pd.Series(d.obs['size_factors'].values if 'size_factors' in d.obs else d.obsm['size_factors'], index=c.index).to_csv(sys.argv[2])
            print(pydeseq2.__version__)
        ''')
        (td / 'run.py').write_text(code)
        clean_env = {k: v for k, v in os.environ.items() if k != 'MPLBACKEND'}   # the notebook kernel's inline backend breaks matplotlib in the other environment
        p = subprocess.run([str(venv_py), '-I', str(td / 'run.py'), str(td / 'c.csv'), str(td / 'sf.csv')], capture_output=True, text=True, timeout=900, env=clean_env)
        if p.returncode == 0:
            sf_ref = pd.read_csv(td / 'sf.csv', index_col=0).iloc[:, 0]
            diff = np.abs(sf_ref.loc[sf.index] / sf - 1)
            check = {'status': 'ran', 'pydeseq2': p.stdout.strip().splitlines()[-1], 'max_relative_difference': float(diff.max()), 'median_relative_difference': float(diff.median())}
        else:
            check = {'status': 'failed', 'stderr': p.stderr[-800:]}
print('Check against PyDESeq2:', json.dumps(check))

norm.T.rename_axis('gene_id').to_csv(PROC / f'{DEV}_norm_counts.tsv.gz', sep='\t')
pd.DataFrame({'size_factor': sf}).rename_axis('sample_id').to_csv(PROC / f'{DEV}_size_factors.csv')

## External cohorts: scale handling and id harmonization
- **GSE309666:** the table is Salmon-derived with Ensembl v113 ids. Median-of-ratios is fit within the cohort without using passage.
- **GSE178514:** FPKM only, so median-of-ratios does not apply. FPKM is rescaled exactly to TPM within the table. Its denominator is the mRNA table, a limit that is carried forward.
- **Ids:** GSE178514 Entrez ids map to Ensembl through NCBI gene_info, with a fallback through symbols in the GSE309666 annotation. This uses annotation only, not expression.

In [ ]:
asc, asc_ann = sg.load_as_deposited(ASC)
asc_ann = asc_ann.copy()
asc_filtered = sg.author_gene_filter(asc)
asc_norm, asc_sf, asc_ref = sg.mor_normalize(asc_filtered)
print(f'{ASC}: {asc.shape[1]} genes -> {asc_filtered.shape[1]} after author filter; size factors {asc_sf.min():.3f}-{asc_sf.max():.3f}; genes non-zero in all samples {len(asc_ref)}')
asc_norm.T.rename_axis('gene_id').to_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t')

bm, bm_ann = sg.load_as_deposited(BM)
idmap = pd.read_csv(PROC / f'{BM}_gene_id_map.tsv', sep='\t', dtype=str).set_index('gene_id')
sym2ens = asc_ann.dropna(subset=['symbol']).drop_duplicates('symbol', keep=False).set_index('symbol').ensembl
idmap['ensembl_final'] = idmap.ensembl.where(idmap.ensembl.notna(), idmap.symbol.map(sym2ens))
bm_tpm = sg.tpm_from_fpkm(bm)
with_ens = idmap.ensembl_final.reindex(bm_tpm.columns)
print(f'{BM}: {bm.shape[1]} ids; Ensembl via gene_info {int(idmap.ensembl.notna().sum())}; after symbol fallback {int(idmap.ensembl_final.notna().sum())}; duplicated Ensembl {int(with_ens.dropna().duplicated().sum())}')
bm_ens = bm_tpm.loc[:, with_ens.notna()].T.groupby(with_ens.dropna().values).sum().T
print(f'  matrix by Ensembl: {bm_ens.shape}. Duplicates are summed, since transcripts of one gene add on the TPM scale.')
bm_ens.T.rename_axis('gene_id').to_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t')
bm_sym = bm_tpm.loc[:, idmap.symbol.reindex(bm_tpm.columns).notna()]
idmap.reset_index().to_csv(PROC / f'{BM}_gene_id_map_final.tsv', sep='\t', index=False)

common = {
    'dev_filtered_genes': filtered.shape[1],
    'asc_filtered_genes': asc_filtered.shape[1],
    'bm_ensembl_genes': bm_ens.shape[1],
    'dev_and_asc': len(set(filtered.columns) & set(asc_filtered.columns)),
    'dev_and_bm': len(set(filtered.columns) & set(bm_ens.columns)),
    'all_three': len(set(filtered.columns) & set(asc_filtered.columns) & set(bm_ens.columns)),
}
print(common)
json.dump({'pydeseq2_check': check, 'gene_overlap': common}, open(MANIFESTS / 'preprocess_report.json', 'w'), indent=1)

## Analysis protocol, written before any MSC score
This is written to a file and hashed so later notebooks can show it wasn't changed after seeing external results.

In [ ]:
protocol = """# SenGuard-MSC analysis protocol

Fixed on 2026-11-01, before any senescence score was computed on an MSC cohort.
It follows G1 and the endpoint the project lead approved.

## Endpoints
1. Development (GSE284666): induced senescence vs control, 14 cell types. Unit of held-out validation: cell type.
2. Primary MSC endpoint (GSE309666): within-donor change in the published SenCat score across passage.
3. Supporting MSC evidence (GSE178514): same statistic in three putative donors, FPKM-derived.
4. Labelled MSC check: the BMMSC cell type held out of development folds. This is one cell line, so it is a case, not a population estimate.

Passage is not a label. No AUROC, accuracy or calibration is computed against passage.

## Model selection rule (development data only)
- Metric: mean over held-out cell types of the within-cell-type AUROC from nested leave-one-cell-type-out.
- All fold-dependent steps (reference, gene filters, feature selection, tuning, threshold) use training cell types only.
- Choice: highest mean. Any candidate within 0.01 of the best counts as tied, and the simplest wins, in the order
  logreg_l1, elastic_net, linear_svm, rbf_svm, random_forest, xgboost.
- Weights and models are saved with hashes before notebook 03 loads MSC scores.

## External statistics
- Per donor: slope of score per passage (primary), Spearman rho, last minus first passage.
- Across donors: mean slope with an exact two-sided sign-flip test. Five donors give a smallest possible p of 0.0625 and three give 0.25,
  so no conventional significance is reachable by design; effect sizes and direction are the result.
- Competitive null: 1,000 draws shuffling weights among the same genes, and 1,000 draws of random expressed genes. Two-sided empirical p.
- Primary score: published SenCat weights. All other scores (hUSI, SenMayo, Hallmark proliferation, refit models, panels) are secondary and reported without selection.
- Independence: refit weights that exclude BMMSC, and exclude BMMSC plus preadipocytes, are the cell-type-independent sensitivity analysis.

## Handling
- External samples are never removed on score grounds; GSM9271440 (donor B p7) stays in, with a sensitivity analysis excluding it.
- Thresholds are never applied to external cohorts, because no labels exist to justify them. Model outputs there are decision values, not probabilities.
- Null and negative results get reported the same way as positive ones.
"""
versioned = REPORTS / 'ANALYSIS_PROTOCOL.md'
if versioned.exists() and versioned.read_text(encoding='utf-8') != protocol:
    raise RuntimeError('ANALYSIS_PROTOCOL.md already exists with different text. Protocol changes must be made and recorded by hand, not overwritten on rerun.')
versioned.write_text(protocol, encoding='utf-8')
print('Protocol sha256', sg.sha256(versioned))

In [ ]:
outs = [PROC / f'{DEV}_labels.csv', PROC / f'{DEV}_norm_counts.tsv.gz', PROC / f'{DEV}_size_factors.csv', PROC / f'{ASC}_norm_counts.tsv.gz', PROC / f'{BM}_tpm_ensembl.tsv.gz',
        PROC / f'{BM}_gene_id_map_final.tsv', MANIFESTS / 'preprocess_report.json', versioned]
print(sg.output_manifest(outs).assign(path=lambda d: d.path.str.replace(str(ROOT), '', regex=False))[['path', 'bytes']].to_string())